# Lab 1 · Notebook 03 — Exploratory Data Analysis

**Course:** Artificial Intelligence with Python  
**Estimated duration:** 60 minutes  
**Prerequisites:** Notebooks 01 and 02, local Jupyter environment with Pandas, NumPy and Matplotlib

## Learning outcomes
By the end of this notebook, you should be able to:

1. **Describe** the shape, data types, summary statistics, and missing values in a dataset.
2. **Visualize** numerical and categorical variables with suitable plots.
3. **Investigate** relationships between features and potential prediction targets.
4. **Distinguish** observations from conclusions and identify questions for preprocessing.

**Dataset:** `data/apartments.csv`, containing 150 **fictional** apartment listings. The data is synthetic and must not be used to draw conclusions about any real housing market.

**Working with exercises:** Complete the code cells marked `# TODO`. Write short interpretations in the Markdown answer spaces. To restart cleanly, choose **Kernel → Restart Kernel and Run All Cells** after completing your exercises.

## 1. Load the dataset and inspect its structure (8 min)

Exploratory Data Analysis (EDA) is the process of understanding data **before** choosing and training a machine-learning model. Here we ask: *What do we have, what is unusual, and what might be predictable?*

Launch JupyterLab from the course repository and open this notebook in `lab01/`, so the relative CSV path below works. The notebook deliberately does not modify the source CSV.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

csv_path = Path("data/apartments.csv")
if not csv_path.exists():
    raise FileNotFoundError(
        f"Could not find {csv_path.resolve()}. "
        "Place this notebook in lab01/ alongside the data/ folder."
    )

df = pd.read_csv(csv_path)
print(f"Dataset shape: {df.shape[0]} rows × {df.shape[1]} columns")
display(df.head())

In this dataset, each row is a listing. The column `monthly_rent_eur` is a possible **regression target**; the other measured characteristics could be **features**. `apartment_id` is an identifier, not a meaningful numeric feature.

A numerical variable (e.g. `area_m2`) can have a distribution; a categorical variable (e.g. `district`) has distinct groups. These require different visualizations.

In [ ]:
print("Data types:")
print(df.dtypes)

print("\nMissing values by column:")
print(df.isna().sum().sort_values(ascending=False))

print("\nSummary of numerical columns:")
display(df.describe().round(2))

### Exercise 1 — First impressions (3 min)

1. How many different districts occur in the dataset? Use `.nunique()`.
2. Which numerical column has the largest range (`max - min`)? Ignore `apartment_id`.
3. Which columns have missing values?

Try the code yourself, then write **two sentences** about the data quality.

In [ ]:
# TODO: count unique districts
# TODO: calculate range (max - min) for numerical columns
# TODO: list columns with missing values

**Your interpretation:**

- 
-

## 2. Understand numerical distributions (12 min)

A **histogram** shows how often values fall within numeric intervals (bins). It is useful for finding a typical range, skewness, or unusually high/low values. The number of bins changes how the same data appears.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(df["monthly_rent_eur"].dropna(), bins=12, edgecolor="white")
ax.set_title("Distribution of monthly rents")
ax.set_xlabel("Monthly rent (EUR)")
ax.set_ylabel("Number of apartments")
plt.tight_layout()
plt.show()

**Mean versus median:** The mean is sensitive to extreme observations; the median is more robust. Inspect both before deciding how to summarize a variable.

In [ ]:
rent = df["monthly_rent_eur"]
print(f"Mean monthly rent:   €{rent.mean():.0f}")
print(f"Median monthly rent: €{rent.median():.0f}")
print(f"Minimum / maximum:   €{rent.min():.0f} / €{rent.max():.0f}")

fig, ax = plt.subplots(figsize=(7, 2.5))
ax.boxplot(rent.dropna().to_numpy(), orientation='horizontal')
ax.set_title("Rent distribution (box plot)")
ax.set_xlabel("Monthly rent (EUR)")
plt.tight_layout()
plt.show()

A **box plot** summarizes the median, quartiles and possible outliers. Points beyond the whiskers are *potential* outliers—not automatically errors. In a real dataset, investigate them before deleting records.

### Exercise 2 — Explore apartment sizes (5 min)

1. Plot a histogram of `area_m2` with **15 bins** and meaningful labels.
2. Print its median and mean.
3. Answer: Does the distribution look symmetric or skewed? Does the choice of bins affect your interpretation?

In [ ]:
# TODO: histogram for area_m2 (15 bins)
# TODO: mean and median of area_m2
# TODO: add title and axis labels; call plt.show()

**Your interpretation:**

-

## 3. Explore categories (10 min)

A **bar chart** is usually better than a histogram for categorical variables. Counts show how many observations belong to each category; averages compare numeric outcomes within categories.

In [ ]:
district_counts = df["district"].value_counts().sort_index()
print("Listings by district:")
display(district_counts)

fig, ax = plt.subplots(figsize=(7, 4))
district_counts.plot(kind="bar", ax=ax)
ax.set_title("Apartment listings by district")
ax.set_xlabel("District")
ax.set_ylabel("Number of listings")
ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
mean_rent_by_district = (
    df.groupby("district")["monthly_rent_eur"]
    .mean()
    .sort_values()
)
display(mean_rent_by_district.round(0))

fig, ax = plt.subplots(figsize=(7, 4))
mean_rent_by_district.plot(kind="barh", ax=ax)
ax.set_title("Mean monthly rent by district")
ax.set_xlabel("Mean monthly rent (EUR)")
ax.set_ylabel("District")
plt.tight_layout()
plt.show()

**Important:** Differences in average rent by district are not automatically caused by district. Apartment size, age, and other characteristics may differ between districts. EDA reveals patterns but does not establish causality.

### Exercise 3 — Is a balcony associated with rent? (4 min)

1. Count listings with and without balconies.
2. Plot mean `monthly_rent_eur` grouped by `balcony`.
3. Why is a simple difference in group means *not enough* to conclude that adding a balcony raises rent?

In [ ]:
# TODO: print counts by balcony
# TODO: calculate and plot mean rent by balcony
# TODO: label the axes and show the plot

**Your interpretation:**

-

## 4. Investigate relationships between numerical variables (12 min)

A **scatter plot** shows individual observations as points. It can suggest a relationship between two numerical variables. Large values on one axis appearing alongside large values on the other indicates a possible positive association.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.scatter(df["area_m2"], df["monthly_rent_eur"], alpha=0.65)
ax.set_title("Does apartment size relate to rent?")
ax.set_xlabel("Area (m²)")
ax.set_ylabel("Monthly rent (EUR)")
plt.tight_layout()
plt.show()

**Correlation** summarizes the *strength and direction of a linear association* between numerical variables. Pearson's correlation lies between -1 and +1. A value near 0 means little **linear** association, not necessarily no relationship at all. Pandas `.corr()` excludes missing pairs by default.

In [ ]:
numeric_features = [
    "area_m2", "rooms", "building_age_years",
    "distance_to_center_km", "monthly_rent_eur"
]
corr = df[numeric_features].corr()
print("Correlation with monthly rent:")
display(corr["monthly_rent_eur"].sort_values(ascending=False).round(2))

fig, ax = plt.subplots(figsize=(7, 4.5))
image = ax.imshow(corr, vmin=-1, vmax=1, cmap="coolwarm")
ax.set_xticks(range(len(numeric_features)), numeric_features, rotation=45, ha="right")
ax.set_yticks(range(len(numeric_features)), numeric_features)
ax.set_title("Pearson correlations between numerical features")
fig.colorbar(image, ax=ax, label="Pearson correlation")
plt.tight_layout()
plt.show()

### Exercise 4 — Distance versus rent (5 min)

1. Make a scatter plot of `distance_to_center_km` against `monthly_rent_eur`.
2. Calculate their Pearson correlation using Pandas.
3. Describe its direction and whether the scatter plot supports your interpretation.
4. Is it appropriate to claim that changing the distance *causes* rent to change? Explain.

In [ ]:
# TODO: scatter plot of distance_to_center_km vs monthly_rent_eur
# TODO: correlation for these two columns
# TODO: axis labels, title, plt.show()

**Your interpretation:**

-

## 5. Missing data and suspicious observations (8 min)

Before training models, identify what is missing and whether measurements appear plausible. **Do not delete or replace values yet:** we will study preprocessing systematically in Lab 2.

Missing value counts alone do not show which rows are affected. The examples below locate those rows for inspection.

In [ ]:
missing_counts = df.isna().sum()
display(missing_counts[missing_counts > 0].rename("missing_count"))

incomplete_rows = df[df.isna().any(axis=1)]
print(f"Rows with at least one missing value: {len(incomplete_rows)}")
display(incomplete_rows.head(8))

# Simple sanity checks (not automatic error detection)
print("Non-positive area:", (df["area_m2"] <= 0).sum())
print("Non-positive rent:", (df["monthly_rent_eur"] <= 0).sum())
print("Duplicate apartment IDs:", df["apartment_id"].duplicated().sum())

### Exercise 5 — Inspect incomplete records (3 min)

1. What percentage of the 150 listings is incomplete (missing at least one field)?
2. Which affected feature would you investigate first, and why?
3. Name one possible risk of simply dropping every incomplete row.

In [ ]:
# TODO: compute percentage of rows with at least one missing value
# TODO: inspect affected columns/rows if useful

**Your interpretation:**

-

## 6. Mini-challenge — Tell a data story (8 min)

Choose **one** question below (or formulate your own), and produce a short evidence-based finding.

- **A.** Are larger apartments more expensive *per square metre*?
- **B.** Does average rent differ between old and new buildings?
- **C.** Is apartment size distributed differently across districts?

Your investigation should include:

1. A **calculation or summary table** supporting your choice.
2. At least **one clearly labeled visualization**.
3. A **3–4 sentence conclusion** describing the pattern, a limitation, and one preprocessing decision needed before model training.

Hints: create `rent_per_m2 = monthly_rent_eur / area_m2`, use `.groupby()`, `.median()`, or a scatter plot. Keep the source CSV unchanged.

In [ ]:
# TODO: choose one of A/B/C
# TODO: make your summary table or calculation
# TODO: generate and label your visualization

**Mini-challenge conclusion:**

1. **Question investigated:**
2. **Observed pattern:**
3. **Limitation or alternative explanation:**
4. **Preprocessing step to consider for Lab 2:**

## Wrap-up — What should an ML practitioner do next?

You have completed the first stages of a typical workflow:

**CSV data → inspect types and missingness → summarize → visualize → question relationships.**

For the next lab, we will turn this imperfect tabular dataset into data suitable for machine learning. We will consider:

- missing-value strategies;
- encoding categorical variables (`district`, `balcony`, `energy_rating`);
- feature scaling and choosing useful predictors;
- separating training and testing data **before fitting preprocessing transformations** to avoid data leakage.

**Before you finish:** Save your notebook with your plots, TODO solutions, and written interpretations. No model training is required in Lab 1.

**Reflection:** Which two dataset features do you expect would help predict monthly rent? What evidence supports that expectation?